# 4c. Forced response figures: the whole distribution, additivity and zonal means

The figures for the forced response, drawn from what notebook 03 saved, so nothing slow runs here. Notebook 03 explains each method, step by step. The figures are in four notebooks, so that none grows too big to open; each opens only the saved results it uses, so they can be run in any order after notebook 03:

| Notebook | Sections |
|---|---|
| 04a | 1. Mean change; 2. Signal to noise (and a check of every model's headline numbers) |
| 04b | 3. Quantile response; 4. Change in width, and which tail |
| 04c | 5. The whole distribution; 6. Additivity; 7. Zonal means and the sea-ice edge |
| 04d | 8. Probability ratios; 9. The shift-only ratio |

The sections keep one numbering across the four.

Every section shows its result in the same four figures, with the four seasons across every map:

1. **What goes in.** The quantities the result is made from, then the result, one row each: one model and experiment (`MODEL`, `EXPERIMENT`).
2. **One model.** The parts and the result for every experiment of `MODEL`.
3. **Every model.** The result for every model (rows), one figure per experiment, dotted where it is not significant.
4. **How many models.** The multi-model median of the result (top row), then at each grid point how many models have a significant increase and how many a significant decrease; one figure per experiment.

This notebook:

| Section | Result | Test |
|---|---|---|
| 5. The whole distribution | where the mean and the width changed (after Bracegirdle et al., 2024) | sections 1 and 4 |
| 6. Additivity | historical against the sum of the single forcings | none |
| 7. Zonal means and the sea-ice edge | sections 1 and 4 by latitude (after Bracegirdle et al., 2024) | sections 1 and 4, zonally averaged |

**Reading the figures.** A quantity has the same colour scale in every figure (set once, under *Colour scales*). Every change is red for an increase and blue for a decrease, and dots mark where a change is not significant. The black lines on the maps of section 5 are the sea-ice edge (the 15% contour of the ensemble-mean concentration over the final 21 years, from notebook 09): hist-nat's dashed, the experiment's solid; the multi-model figures draw the edge of the models' median concentration. Without the sea ice they are left out. In the counts, the colour changes where more than half the models agree: reds turn brown (an increase) and blues purple (a decrease). The paper figures, which put the models together, are in notebook 06. The cells before each figure pick out its data and show them.

Names end in their type: `_tree` (DataTree), `_ds` (Dataset), `_da` (DataArray), `_arr` (numpy array), `_df` (DataFrame). Each saved result has the same name as in notebook 03. Until 14 October 2026 the four were one notebook, `04_forced_response_figures.ipynb` (in the git history); the version before that, from 7 October, is in `archive/forced_response_figures_2026-10-07.ipynb`.

## Setup

In [ ]:
import sys
sys.path[:0] = ['..']   # this repository: the extant package and plotting_modules

import logging

import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr

from plotting_modules import formatting
from extant import config, loading, paths, preprocessing as pp, response_change as rc, sea_ice
from extant import significance as sig, storage, zonal
from extant.logger import setup_logging
from extant.plots import forced_response as fr, response_change as rcp, sea_ice as sip, zonal as zp

logger = setup_logging('extant', logging.INFO)
xr.set_options(display_expand_attrs=False, display_expand_data=False)
pd.set_option('display.max_rows', 200)   # every model, experiment and season in one table

In [ ]:
%matplotlib inline

## Settings

In [ ]:
VARIABLE = 'tas'
var = config.VARIABLES[VARIABLE]
UNITS = var.units

WINDOW = config.WINDOW
LAST_FULL_YEAR = config.LAST_FULL_YEAR

#(t): The experiments, in the order of every figure, and the seasons, the columns of every map
EXPERIMENTS = ['hist-GHG', 'hist-aer', 'hist-totalO3', 'historical']
SEASONS = ['DJF', 'MAM', 'JJA', 'SON']

#(t): The model and experiment of the "what goes in" and "one model" figures
MODEL = config.PLOT_SEL['model']
EXPERIMENT = config.PLOT_SEL['experiment']

#(t): Significance level of every test
ALPHA = 0.05


def result_path(name):
    """Where notebook 03 saved a result: results/<variable>/<run>/<name>.zarr."""
    return paths.result(name, VARIABLE, RUN)

### Colour scales

One scale per quantity, used by every figure that shows it (here and in notebooks 05 and 06), so the same colour always means the same value. Each is a `fr.Scale`: the contour levels, the colormap and the colour-bar label. Their ranges, per variable, are in `fr.RANGES`; notebook 06 (*Check the colour ranges*) says how much of each field falls beyond them.

In [ ]:
scales = fr.colour_scales(VARIABLE, alpha=ALPHA)

In [ ]:
#(t): What each one covers
pd.DataFrame({name: {'from': scale.levels[0], 'to': scale.levels[-1], 'colormap': getattr(scale.cmap, 'name', scale.cmap),
                     'label': scale.label}
              for name, scale in vars(scales).items()}).T

## Open the data and the saved results

Run **one** of the next two sections. Both leave `lesfmip_season_tree` (the seasonal means, used only for the figures at one grid point in 5.5) and `RUN`, which says which saved results to open.

### Sample data

In [ ]:
#(t): The sample's monthly means, as a DataTree /<model>/<experiment>
lesfmip_sample_tree = loading.open_lesfmip_sample(experiments=config.MAIN_EXPERIMENTS)
lesfmip_sample_tree

In [ ]:
#(t): Seasonal means in analysis units, through the same pre-processing as notebook 01
lesfmip_season_tree = pp.seasonal_tree(lesfmip_sample_tree, VARIABLE).compute()
RUN = 'sample'

### Full data (JASMIN)

In [ ]:
#(t): The seasonal means notebook 01 saved: DATA_DIR is read first, then SCRATCH
seasonal_path = paths.find(paths.seasonal('lesfmip', VARIABLE))
seasonal_path

In [ ]:
lesfmip_season_tree = xr.open_datatree(seasonal_path, engine='zarr', chunks={})
RUN = 'full'

In [ ]:
#(t): Only the experiments analysed (filter keeps the model nodes above them)
lesfmip_season_tree = lesfmip_season_tree.filter(lambda node: node.name in config.MAIN_EXPERIMENTS)
lesfmip_season_tree

### The saved results

Everything sections 5, 6 and 7 use, from `results/<variable>/<RUN>/` under `paths.DATA_DIR` or `paths.SCRATCH` (the log gives each full path), loaded into memory. Grouped by the section that uses them.

In [ ]:
#(t): Section 5: hist-nat's and every experiment's final-years mean and quantiles, the tests of the mean and the
#(t): width, the tails, and S/N (for the 'emerged' and 'robust' tests of the mean)
mean_bootstrap_ds = storage.open_dataset(result_path('mean_bootstrap'), load=True)
hist_nat_final_q_da = storage.open_dataarray(result_path('hist_nat_final_quantiles'), load=True)
final_q_da = storage.open_dataarray(result_path('final_quantiles'), load=True)
qrange_change_ds = storage.open_dataset(result_path('qrange_change'), load=True)
tails_ds = storage.open_dataset(result_path('tails'), load=True)
lesfmip_seasonal_sn_ds = storage.open_dataset(result_path('sn'), load=True)

#(t): Sections 6 and 7: each experiment's change from its own baseline, and the zonal-mean tests
own_changes_da = storage.open_dataarray(result_path('own_changes'), load=True)
zonal_mean_ds = storage.open_dataset(result_path('zonal_mean_test'), load=True)
zonal_width_ds = storage.open_dataset(result_path('zonal_width_test'), load=True)

In [ ]:
#(t): What each result holds: its dims and its variables
results = {
    'mean_bootstrap_ds': mean_bootstrap_ds, 'hist_nat_final_q_da': hist_nat_final_q_da, 'final_q_da': final_q_da,
    'qrange_change_ds': qrange_change_ds, 'tails_ds': tails_ds, 'lesfmip_seasonal_sn_ds': lesfmip_seasonal_sn_ds,
    'own_changes_da': own_changes_da, 'zonal_mean_ds': zonal_mean_ds, 'zonal_width_ds': zonal_width_ds,
}
pd.DataFrame({
    name: {'dims': dict(result.sizes), 'variables': ', '.join(result.data_vars) if isinstance(result, xr.Dataset) else ''}
    for name, result in results.items()
}).T

In [ ]:
#(t): The number of models: the top of the colour bar of every count figure
N_MODELS = mean_bootstrap_ds.sizes['model']
N_MODELS

### Sea ice

The sea-ice edge on the maps (sections 5 and 7) and the zonal means (section 7) comes from notebook 09, section 3. Run **one** of the next two sections. Both leave `ice_extents_tree` (each member's seasonal extent), `ice_climatology_da` (the ensemble-mean concentration over the final years), `ice_edges_da` (the equivalent latitude of the edge in the final years), and `model_ice_da` and `median_ice_da` (`MODEL`'s concentration, and the models' median, for the edges of the one-model and multi-model maps); without sea ice they are `None` and the figures leave the edge out.

#### Without sea ice

Before notebook 09 has saved the sea ice (on the sample, that is when the notebooks are run in order).

In [ ]:
ice_extents_tree = ice_climatology_da = ice_edges_da = model_ice_da = median_ice_da = None

#### With sea ice (JASMIN)

In [ ]:
#(t): What notebook 09 saved, for the same RUN (on the sample too, once notebook 09 has run on it)
ice_extents_tree = storage.open_tree(paths.result('extent', 'siconc', RUN), load=True)
ice_climatology_da = storage.open_dataarray(paths.result('climatology', 'siconc', RUN), load=True)
ice_edges_da = sea_ice.edge_latitude(ice_extents_tree, years=WINDOW)
model_ice_da = ice_climatology_da.sel(model=MODEL)
median_ice_da = ice_climatology_da.median('model')

## 5. The whole distribution: where the mean and the width have changed

Sections 1, 3 and 4 together, in the form of Bracegirdle et al. (2024, Fig. 1): the change in the low end of the distribution (Q05), its middle (the mean), its high end (Q95) and its width (Q95 − Q05). Equal changes in Q05, the mean and Q95 are a pure shift; a bigger change in Q05 than in Q95 is a distribution narrowing from below. (The values behind these changes are in 1.1, 3.1 and 4.1, in notebooks 04a and 04b.)

Then each grid point is put in one of nine classes: the mean significantly down, unchanged or up (section 1's test), times the width significantly narrower, unchanged or wider (section 4's test). The 3 × 3 key reads like a table: red is a warmer mean, blue a cooler one; yellow mixed in is a narrower distribution, purple a wider one.

`MEAN_TEST` chooses the test for the mean: `'significant'` is the hist-nat bootstrap, as in section 1; `'robust'` asks for $|\mathrm{S/N}| \ge 2$ as well (section 2, notebook 04a), and `'emerged'` for that alone.

In [ ]:
#(t): Which test says the mean has changed: 'significant' (hist-nat bootstrap, section 1), 'emerged' (S/N) or 'robust' (both)
MEAN_TEST = 'significant'

In [ ]:
#(t): The year of section 2 (notebook 04a): the centre of the final WINDOW years
sn_year = int(lesfmip_seasonal_sn_ds['year'][LAST_FULL_YEAR])
sn_year

In [ ]:
#(t): S/N in the year of section 2, for the 'emerged' and 'robust' tests
sn_final_da = lesfmip_seasonal_sn_ds[VARIABLE].sel(year=sn_year)
sn_final_da

In [ ]:
#(t): Every change and its test in one Dataset on (model, experiment, season, lat, lon)
change_ds = rc.change_summary(mean_bootstrap_ds['mean_change'], mean_bootstrap_ds['pvalue'], sn_final_da, qrange_change_ds,
                              tails_ds, alpha=ALPHA)
change_ds

In [ ]:
#(t): hist-nat's distribution over the final WINDOW years: Q05, the mean and Q95, and the width of its internal
#(t): variability (as tested in section 4)
hist_nat_fields_ds = xr.Dataset({
    'Q05': hist_nat_final_q_da.sel(quantile=0.05, drop=True),
    'Mean': mean_bootstrap_ds['hist_nat_mean'],
    'Q95': hist_nat_final_q_da.sel(quantile=0.95, drop=True),
    'Q95 − Q05': qrange_change_ds['hist_nat_qrange'],
})
hist_nat_fields_ds

In [ ]:
#(t): Every experiment's, over its final WINDOW years
experiment_fields_ds = xr.Dataset({
    'Q05': final_q_da.sel(quantile=0.05, drop=True),
    'Mean': mean_bootstrap_ds['experiment_mean'],
    'Q95': final_q_da.sel(quantile=0.95, drop=True),
    'Q95 − Q05': qrange_change_ds['experiment_qrange'],
})
experiment_fields_ds

In [ ]:
#(t): The change in each: the same as sections 1, 3 and 4
change_fields_ds = experiment_fields_ds - hist_nat_fields_ds
change_fields_ds

In [ ]:
#(t): Which changes are significant: the mean by MEAN_TEST, the width by the hist-nat bootstrap (Q05 and Q95 have no test)
change_significant_ds = xr.Dataset({'Mean': change_ds[f'mean_{MEAN_TEST}'], 'Q95 − Q05': change_ds['width_significant']})
change_significant_ds

In [ ]:
#(t): Not significant, where there is data
change_not_significant_ds = ~change_significant_ds & change_fields_ds[list(change_significant_ds)].notnull()

In [ ]:
#(t): The sign of each significant change: +1 up, -1 down, 0 not significant
mean_sign_da = sig.significant_sign(change_fields_ds['Mean'], change_significant_ds['Mean'])
width_sign_da = sig.significant_sign(change_fields_ds['Q95 − Q05'], change_significant_ds['Q95 − Q05'])

In [ ]:
#(t): The joint class, 0 to 8, in the order of the 3 x 3 key
joint_class_da = rc.joint_class(mean_sign_da, width_sign_da)
joint_class_da

### 5.1 What goes in

The change in each part of the distribution, `EXPERIMENT` against hist-nat for `MODEL`, dotted where the mean or the width change is not significant; then the class the two give. The three changes of the distribution share one colour scale; the width has its own.

In [ ]:
#(t): MODEL and EXPERIMENT, every season: the change
model_change_fields_ds = change_fields_ds.sel(model=MODEL, experiment=EXPERIMENT, season=SEASONS)
model_change_fields_ds

In [ ]:
#(t): Where the change is not significant
model_change_not_significant_ds = change_not_significant_ds.sel(model=MODEL, experiment=EXPERIMENT, season=SEASONS)

In [ ]:
#(t): The joint class
model_joint_class_da = joint_class_da.sel(model=MODEL, experiment=EXPERIMENT, season=SEASONS)

In [ ]:
panels = fr.row_grid(
    [fr.Row(model_change_fields_ds['Q05'], scales.change, 'ΔQ05', edges=sip.edge_pair(model_ice_da, EXPERIMENT)),
     fr.Row(model_change_fields_ds['Mean'], scales.change, 'Δ mean',
            not_significant=model_change_not_significant_ds['Mean'], edges=sip.edge_pair(model_ice_da, EXPERIMENT)),
     fr.Row(model_change_fields_ds['Q95'], scales.change, 'ΔQ95', edges=sip.edge_pair(model_ice_da, EXPERIMENT)),
     fr.Row(model_change_fields_ds['Q95 − Q05'], scales.spread_change, 'Δ(Q95 − Q05)',
            not_significant=model_change_not_significant_ds['Q95 − Q05'],
            edges=sip.edge_pair(model_ice_da, EXPERIMENT)),
     fr.Row(model_joint_class_da, scales.joint, 'What changed', edges=sip.edge_pair(model_ice_da, EXPERIMENT))],
    title=f'The distribution\'s change: {MODEL}, {EXPERIMENT}, final {WINDOW} years against hist-nat',
    tag=True,
    not_significant_label=f'Not significant (p ≥ {ALPHA:g})',
)

### 5.2 One model

What changed, for every experiment of `MODEL` (1.2 and 4.2, in notebooks 04a and 04b, show the mean and the width changes themselves).

In [ ]:
#(t): MODEL's joint class, every experiment and season
model_forcing_class_da = joint_class_da.sel(model=MODEL, experiment=EXPERIMENTS, season=SEASONS)
model_forcing_class_da

In [ ]:
panels = fr.map_grid(
    model_forcing_class_da, scales.joint,
    title=f'{MODEL}: where the mean and the width have changed, final {WINDOW} years against hist-nat',
    row_dim='experiment', col_dim='season',
    edges=sip.edge_pair(model_ice_da, EXPERIMENTS),
)

### 5.3 Every model

The joint class of every model; one figure per experiment.

In [ ]:
#(t): The joint class for every model, experiment and season
all_class_da = joint_class_da.sel(experiment=EXPERIMENTS, season=SEASONS)
all_class_da

In [ ]:
for experiment in EXPERIMENTS:
    panels = fr.map_grid(
        all_class_da.sel(experiment=experiment), scales.joint, col_dim='season',
        title=f'Where the mean and the width have changed, every model: {experiment}',
        edges=sip.edge_pair(ice_climatology_da, experiment),
    )

### 5.4 How many models

The multi-model median of the mean change and of the width change (the top two rows), then the number of models where the mean has changed and the distribution has widened, or narrowed. One figure per experiment.

In [ ]:
#(t): The mean change for every model, experiment and season (as in section 1, notebook 04a)
all_mean_change_da = mean_bootstrap_ds['mean_change'].sel(experiment=EXPERIMENTS, season=SEASONS)
all_mean_change_da

In [ ]:
#(t): Its multi-model median: the top row of the figures below
median_mean_change_da = all_mean_change_da.median('model')
median_mean_change_da

In [ ]:
#(t): The change in width for every model, experiment and season (as in section 4, notebook 04b)
all_width_change_da = qrange_change_ds['qrange_change'].sel(experiment=EXPERIMENTS, season=SEASONS)
all_width_change_da

In [ ]:
#(t): Its multi-model median: the second row
median_width_change_da = all_width_change_da.median('model')
median_width_change_da

In [ ]:
#(t): How many models changed in the mean and became wider, and how many narrower
joint_counts_da = rc.joint_change_counts(change_ds, mean_test=MEAN_TEST)
joint_counts_da

In [ ]:
#(t): Every experiment and season
all_joint_counts_da = joint_counts_da.sel(experiment=EXPERIMENTS, season=SEASONS)

In [ ]:
for experiment in EXPERIMENTS:
    panels = fr.count_grid(
        all_joint_counts_da.sel(experiment=experiment), N_MODELS,
        above=[fr.Row(median_mean_change_da.sel(experiment=experiment), scales.change, 'Median\nmean change',
                      edges=sip.edge_pair(median_ice_da, experiment)),
               fr.Row(median_width_change_da.sel(experiment=experiment), scales.spread_change, 'Median\nwidth change',
                      edges=sip.edge_pair(median_ice_da, experiment))],
        title=f'How many models have changed in both the mean and the width: {experiment}',
        label='Number of models where the mean changed and the distribution narrowed or widened',
        row_dim='change', col_dim='season',
        row_labels={'mean and wider': 'Mean and\nwider', 'mean and narrower': 'Mean and\nnarrower'},
        edges=sip.edge_pair(median_ice_da, experiment),
    )

### 5.5 A closer look at one point

At a single grid point and season, the final years of each experiment (members pooled) against hist-nat's:

- **a)** the two distributions, with arrows from each hist-nat quantile (Q05, Q50, Q95) to the experiment's. Equal arrows are a pure shift;
- **b)** the change in every quantile, for every experiment, with a bootstrap (whole members) 5–95% range. A flat curve is a pure shift; a rising one, the upper tail moving more; a falling one, the lower tail.

The point and season are where the mean and the width both changed and the width changed most (south of 60°S, any season). Set `change_point` to look anywhere else.

In [ ]:
#(t): The point and its season: where the mean and the width both changed, and the width changed most
change_point = rc.strongest_joint_change(change_ds, MODEL, EXPERIMENT, mean_test=MEAN_TEST)
# change_point = dict(season='JJA', lat=-75.0, lon=-180.0)   # or choose one
change_point

In [ ]:
#(t): The experiments MODEL has, hist-nat first, then in the order of the figures
point_experiments = [e for e in ['hist-nat', *EXPERIMENTS] if e in lesfmip_season_tree[MODEL].children]
point_experiments

In [ ]:
#(t): Their final years at that point and season, on (member, year)
point_samples = rc.final_years(lesfmip_season_tree, MODEL, point_experiments, change_point, years=WINDOW,
                               variable=VARIABLE)
point_samples[EXPERIMENT]

In [ ]:
#(t): The change in every quantile of each experiment against hist-nat, with its bootstrap range
point_shifts = {e: rc.quantile_shift(point_samples[e], point_samples['hist-nat']) for e in point_experiments if e != 'hist-nat'}
point_shifts[EXPERIMENT]

In [ ]:
#(t): Quick look: hist-nat's and EXPERIMENT's final years (members pooled), and every experiment's quantile shift
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for name in ('hist-nat', EXPERIMENT):
    axes[0].hist(point_samples[name].values.ravel(), bins=30, alpha=0.5, label=name)
for name, shift_ds in point_shifts.items():
    shift_ds['shift'].plot(x='quantile', ax=axes[1], label=name)
axes[0].set_title('point_samples')
axes[1].set_title("point_shifts[experiment]['shift']")
axes[0].legend()
axes[1].legend();

In [ ]:
panels = rcp.shift_and_widen(
    point_samples, point_shifts, EXPERIMENT,
    test=change_ds.sel(model=MODEL, experiment=EXPERIMENT).sel(change_point),
    units=UNITS,
    title=f'How the distribution changes: {MODEL}, {change_point["season"]} at {formatting.format_latlon(change_point)}',
)

### 5.6 Every model and forcing, as area means

Area means south of 60°S. First every model's change in the mean and in the width for each forcing (dots: one per model; bars: the multi-model median), then the two against each other, then the lower tail's change against the upper tail's (above the dashed 1:1 line the upper tail stretched more).

In [ ]:
#(t): Area means south of 60°S of every change, and the share of the area in each class
change_regional_ds = rc.regional_mean(change_ds.sel(experiment=EXPERIMENTS), lat_max=-60, mean_test=MEAN_TEST)
change_regional_ds

In [ ]:
panels = fr.forcing_dots(
    change_regional_ds,
    {'mean_change': f'Mean change ({UNITS})', 'width_change': f'Change in Q95 − Q05 ({UNITS})'},
    title='Area mean south of 60°S: every model (markers) and the multi-model median (bars)',
    columns=SEASONS, experiments=EXPERIMENTS,
)

In [ ]:
panels = rcp.response_scatter(change_regional_ds, 'mean_change', 'width_change',
                              title='Area mean south of 60°S: change in width against mean change')

In [ ]:
panels = rcp.response_scatter(change_regional_ds, 'lower_tail_change', 'upper_tail_change', diagonal=True,
                              title='Area mean south of 60°S: upper-tail against lower-tail change')

In [ ]:
#(t): The share of the area south of 60°S where neither, the mean only, the width only, or both changed (%)
class_columns = [f'fraction_{name.replace(" ", "_")}' for name in rc.CHANGE_CLASSES.values()]
class_share_df = (100 * change_regional_ds[class_columns].sel(season=SEASONS)).to_dataframe().round(0)
class_share_df

## 6. Additivity: does historical equal the sum of the single forcings?

Each experiment's change from its own pre-industrial baseline: its final 21 years minus its own 1850–1900 (notebook 03, section 6). A single-forcing experiment contains *only* its forcing, so this is its response; measured from hist-nat, as in the other sections, a sum of four forcings would subtract hist-nat's volcanic cooling four times.

$$
\Delta_e = \bar{x}^{\,e}_{\text{final } L \text{ years}} - \bar{x}^{\,e}_{1850\text{–}1900},
\qquad
\text{residual} = \Delta_{\mathrm{historical}} - \left(\Delta_{\mathrm{nat}} + \Delta_{\mathrm{GHG}} + \Delta_{\mathrm{aer}} + \Delta_{\mathrm{O_3}}\right).
$$

The residual holds everything the parts leave out: land use (hist-lu, which should be very small over Antarctica), any other forcing, and non-additive interaction between forcings. There is no test, so the last part is a table rather than a count.

In [ ]:
#(t): Every term: each part, their sum, historical and the residual; and each part's fraction of historical
additivity_ds = rc.additivity(own_changes_da)
additivity_ds

### 6.1 What goes in

Each single forcing's change from its own baseline; `MODEL`.

In [ ]:
#(t): MODEL's single forcings, every season
model_parts_da = additivity_ds['response'].sel(model=MODEL, season=SEASONS, term=list(rc.ADDITIVE_PARTS))
model_parts_da

In [ ]:
panels = fr.map_grid(
    model_parts_da, scales.change,
    title=f'What goes into the additivity: {MODEL}, final {WINDOW} years minus 1850–1900',
    row_dim='term', col_dim='season',
)

### 6.2 One model

The sum of the parts, historical, and the residual for `MODEL`.

In [ ]:
#(t): MODEL's sum of parts, historical and residual, every season
model_additivity_da = additivity_ds['response'].sel(model=MODEL, season=SEASONS,
                                                    term=['sum of parts', 'historical', 'residual'])
model_additivity_da

In [ ]:
panels = fr.map_grid(
    model_additivity_da, scales.change,
    title=f'{MODEL}: historical against the sum of the single forcings',
    row_dim='term', col_dim='season',
)

### 6.3 Every model

One figure each for the sum of the parts, historical, and the residual.

In [ ]:
#(t): Every model's sum of parts, historical and residual, every season
all_additivity_da = additivity_ds['response'].sel(season=SEASONS, term=['sum of parts', 'historical', 'residual'])
all_additivity_da

In [ ]:
for term in all_additivity_da['term'].values:
    panels = fr.map_grid(
        all_additivity_da.sel(term=term), scales.change, col_dim='season',
        title=f'Historical against the sum of the single forcings, every model: {term}',
    )

### 6.4 Every model, as numbers

Area means south of 60°S of every term, and each part's share of historical's area mean (the share of the area means, which does not blow up where historical's change is small at a point).

In [ ]:
#(t): Area means south of 60°S of every term, every season
antarctic_terms_da = sig.area_mean(additivity_ds['response'].sel(season=SEASONS, lat=slice(None, -60)))
antarctic_terms_da

In [ ]:
#(t): As a table: models and seasons down, terms across
antarctic_terms_df = antarctic_terms_da.to_series().unstack('term')[list(additivity_ds['term'].values)].round(2)
antarctic_terms_df

In [ ]:
#(t): Each part's and the residual's share of historical (%)
share_columns = [*rc.ADDITIVE_PARTS, 'residual']
share_of_historical_df = (100 * antarctic_terms_df[share_columns].div(antarctic_terms_df['historical'], axis=0)).round(0)
share_of_historical_df

## 7. Zonal means and the sea-ice edge

After Bracegirdle et al. (2024, *npj Clim. Atmos. Sci.* 7, 276; [doi:10.1038/s41612-024-00822-y](https://doi.org/10.1038/s41612-024-00822-y)), Figs 1–3, 6 and 8, which compare the change in extreme seasons with the change in the background climate, latitude by latitude, and relate the difference to the retreating sea ice. Their methods differ from ours, so these figures keep their layout but show this project's changes:

| Column | Here | The paper |
|---|---|---|
| Mean | $\Delta$ mean (section 1) | the background climate (smoothed ensemble mean) |
| Low extremes | $\Delta(Q_{05} - Q_{50})$ of the internal variability: > 0 where the low extremes move up towards the middle | p10, minus the background |
| High extremes | $\Delta(Q_{95} - Q_{50})$: > 0 where the high extremes move away from the middle | p90, minus the background |
| Width | $\Delta(Q_{95} - Q_{05})$ = high − low (section 4) | p90 − p10 |

**Every model is shown on its own** (the paper averages five). The zonal means are tested (notebook 03, section 8): dots on a line mark a significant change in the mean or the width (both by the hist-nat bootstrap), and the grey band is what hist-nat's own variability gives the width. The dashed lines are the sea-ice edge (equivalent latitude, notebook 09 section 3) in each experiment's final years. These four columns are the paper's, so here the seasons run down the page (7.1, 7.2), or get a figure each (7.3, 7.4).

The paper's key result for temperature: in winter (JJA), over the retreating sea ice, the cold extremes warm much more than the warm extremes, so the distribution narrows. Here that would show as a positive low-extreme change and a negative width change just poleward of the ice edge, in the experiments whose edge has moved.

In [ ]:
#(t): The four columns, zonally averaged, with the zonal tests
zonal_ds = zonal.zonal_summary(change_ds, zonal_mean_ds, zonal_width_ds)
zonal_ds

### 7.1 What goes in

The maps behind the zonal means for `MODEL` and `EXPERIMENT` (the paper's Figs 1 and 3), seasons down, with the 15% ice edge of the experiment (solid) and of hist-nat (dashed). Dotted where the mean or the width change is not significant.

In [ ]:
panels = rcp.extreme_change_maps(
    change_ds, MODEL, EXPERIMENT, seasons=SEASONS, mean_test=MEAN_TEST, ice=ice_climatology_da, units=UNITS,
    mean_levels=scales.change.levels, extreme_levels=scales.spread_change.levels,
    title=f'What goes into the zonal means: {MODEL}, {EXPERIMENT}',
)

### 7.2 One model

`MODEL`'s change at each latitude, every experiment as a line, seasons down.

In [ ]:
#(t): MODEL's four columns, every season, on one 'column' dim
model_zonal_da = zonal_ds[['mean_change', 'low_extreme_change', 'high_extreme_change', 'width_change']].sel(
    model=MODEL, season=SEASONS).to_dataarray('column')
model_zonal_da

In [ ]:
#(t): Quick look: against latitude, one line per experiment; the four columns down, the seasons across
model_zonal_da.plot.line(x='lat', hue='experiment', row='column', col='season', sharey='row');

In [ ]:
fig, axes = zp.zonal_change_grid(zonal_ds, rows='season', lines='experiment', row_values=SEASONS,
                                 select={'model': MODEL}, edges=ice_edges_da, units=UNITS,
                                 title=f'{MODEL}: zonal-mean change, final {WINDOW} years against hist-nat')

### 7.3 Every model

One row per model, the experiments as lines, one figure per season. The mean has its own scale; the extremes and the width share one, so the width column is the high column minus the low.

In [ ]:
for season in SEASONS:
    fig, axes = zp.zonal_change_grid(zonal_ds, season, edges=ice_edges_da, units=UNITS,
                                     title=f'Zonal-mean change, final {WINDOW} years against hist-nat, every model: '
                                           f'{season}')

### 7.4 Every model, by forcing

The same the other way round: one row per experiment and the models as lines, to see how far the models agree on each forcing; one figure per season.

In [ ]:
for season in SEASONS:
    fig, axes = zp.zonal_change_grid(zonal_ds, season, rows='experiment', lines='model', edges=ice_edges_da,
                                     units=UNITS, title=f'Zonal-mean change by forcing, every model as a line: {season}')

### 7.5 Where the seasons vary most

hist-nat's own width $Q_{95} - Q_{05}$ (final 21 years), zonally averaged, with each model's hist-nat ice edge. The paper finds the year-to-year variability of winter temperature largest just poleward of the ice edge, where a season can be over ice or open water; a model whose peak sits elsewhere, or whose edge is far from the others', will put its changes in variability at other latitudes too.

In [ ]:
fig, axes = zp.zonal_reference_width(zonal_ds, seasons=SEASONS, edges=ice_edges_da, units=UNITS,
                                     title='Year-to-year variability in hist-nat: zonal-mean width, every season')

### 7.6 The sea ice itself (JASMIN)

Each experiment's extent through time (ensemble mean and the 5–95% of members), and the equivalent latitude of each model's edge in the final years.

In [ ]:
#(t): Each experiment's extent: the ensemble mean and the 5-95% of members
ice_extent_summary_ds = sea_ice.extent_summary(ice_extents_tree)
ice_extent_summary_ds

In [ ]:
fig, axes = sip.extent_timeseries(ice_extent_summary_ds, seasons=SEASONS,
                                  title='Sea-ice extent, every model, experiment and season')

In [ ]:
#(t): The equivalent latitude of each model's ice edge in the final years
ice_edges_da.to_dataframe().unstack('season')['edge_latitude'].round(1)